In [ ]:
!pip install transformers==4.36.0
!pip install accelerate==0.25.0
!pip install peft==0.7.1
!pip install datasets==2.16.1
!pip install torch==2.1.0

In [ ]:
import torch
import numpy as np
import pandas as pd
from datetime import datetime
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_auc_score, average_precision_score
)

In [2]:
import pandas as pd
train_data = pd.read_csv('train.csv')

In [3]:
train_data

,id,text,score
0,127857,пидорасы наше правительство издеваються над пе...,1
1,86622,чё цена?,0
2,33294,старый пиндосовский прихвостень...,1
3,421,хорошо потрудились... молодец!!!,0
4,118550,хаба это скучная возня по полу👎,0
...,...,...,...
173798,176340,"чё такие глазки?да,да,на вот такой он высоте.с...",0
173799,111515,спасибо за совет обязательно,0
173800,115224,опять иду на пьянку. у внука годовщина свадьбы,0
173801,2974,алла жар болсын! желаю скорейшего освобождения...,0


In [4]:
test_data = pd.read_csv('test.csv')

In [5]:
test_data

,id,text
0,152591,"господь услышал слова людей, наши молитвы дошл..."
1,108771,их тоже надо выстрелами в головы пристрелить.б...
2,198853,"не ужели не понятно, он так завтракает, обедае..."
3,194736,"я открыла, прочитала"
4,88110,нет ты давай по взрослому с помадой :p :d
...,...,...
74482,142159,ну дай то бог чтоб это правда. мож и нам дадут
74483,113769,"я бы с удовольствием работала,да развалили наш..."
74484,60453,"все красавцы, как на подбор"
74485,166036,жаль.только не в чём неповинные мирные люди по...


Воспользуемся популярной моделью с HF для классификации текстов на русском на токсичность

In [ ]:
from transformers import pipeline
classifier = pipeline("text-classification", model="SkolkovoInstitute/russian_toxicity_classifier")

Попробуем проскорить весь датасет без предобработки и расчитатем метрики бинарной классификации и ROC AUC по которому будет проводиться оценивание тестового датасета

In [25]:
text = list(train_data["text"])

results = []
for single_text in tqdm(text, desc="Processing texts"):
    result = classifier(single_text)
    results.append(result[0])

Processing texts: 100%|██████████| 173803/173803 [31:30<00:00, 91.94it/s] 


In [26]:
predicted_labels = [0 if elem['label'] == 'neutral' else 1 for elem in results]

In [27]:
true_labels = list(train_data['score'])

In [28]:
print(classification_report(predicted_labels, true_labels))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99    142869
           1       0.96      0.97      0.96     30934

    accuracy                           0.99    173803
   macro avg       0.97      0.98      0.98    173803
weighted avg       0.99      0.99      0.99    173803



In [29]:
roc_auc_score(predicted_labels, true_labels)

np.float64(0.977729214464079)

На тренировочных данных roc-auc порядка 0.97-0.98, чего недостаточно для нашей задачи

Попробуем дообучить используемую модель через Lora так как это лучший способ по соотношению затрачиваемых ресурсов на дообучения модели и получаемого качества. Будем использовать в качестве базы модель ты же модель от сколтеха так как она уже обучена именно на детекцию токсичности в русском языке и дает нелпохое качество на трейн сете. Нам останется только вложить в нее спецефичность именно нашего датасета с комментариями

фиксируем все рандомстейты

In [3]:
import random

def set_all_seeds(seed=42):
    """Фиксирует все случайные сиды для воспроизводимости"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Все случайности зафиксированы (seed={seed})")

# Устанавливаем seed
SEED = 42
set_all_seeds(SEED)

[16:39:56] Все случайности зафиксированы (seed=42)


Пропишем параметры дообучения с помощью Лора. Так как модель будет дообучаться в коллабе на Т4 воспользуемся батч сайзом именно 16, чтобы влезть по памяти. Также лора ранг оставим равным 8, чтобы также не исчерпать используемый ресурс. Параметр Lr поставим порядка е-4, это больше чем в среднем при полном дообучении и часто используется при fine-tuning чере Лора.

В качестве оптимизатора выберем adamw как стандарт для работы с текстовыми данными и трансформерными моделями. Попробуем дообучиться без использовании шедуреа и ранней остановки, проведем всего 2 эпохи обучения, но на всем объеме трейн данных. Учитывая доступные ресурсы, это кжается оптимальным вариантом

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "SkolkovoInstitute/russian_toxicity_classifier"

BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-4
NUM_EPOCHS = 2
MAX_LENGTH = 256
LORA_R = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
USE_FP16 = False

Определим функции предобработки данных, расчет метрик и обертку для токенизации

In [ ]:
def log_message(message):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {message}")

def prepare_data(train_data):
    log_message("Начинаю подготовку данных...")

    if isinstance(train_data, pd.DataFrame):
        df = train_data
        if 'скор' in df.columns:
            df = df.rename(columns={'скор': 'label'})
        elif 'score' in df.columns:
            df = df.rename(columns={'score': 'label'})
    else:
        df = pd.DataFrame(train_data, columns=['text', 'label'])

    unique_labels = df['label'].unique()
    log_message(f"Уникальные метки в данных: {sorted(unique_labels)}")

    dataset = Dataset.from_pandas(df[['text', 'label']])
    split = dataset.train_test_split(test_size=0.2, seed=42)

    log_message(f"Размер тренировочного набора: {len(split['train'])}")
    log_message(f"Размер валидационного набора: {len(split['test'])}")

    train_labels = split['train']['label']
    val_labels = split['test']['label']

    if len(train_labels) > 0:
        train_dist = np.bincount(train_labels, minlength=2)
        log_message(f"Распределение меток в train: 0={train_dist[0]}, 1={train_dist[1]}")

    if len(val_labels) > 0:
        val_dist = np.bincount(val_labels, minlength=2)
        log_message(f"Распределение меток в val: 0={val_dist[0]}, 1={val_dist[1]}")

    return split['train'], split['test']

def tokenize_function(examples, tokenizer):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH
    )

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='binary')
    precision = precision_score(labels, predictions, average='binary')
    recall = recall_score(labels, predictions, average='binary')

    return {
        "accuracy": float(accuracy),
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall)
    }


In [ ]:
def train_lora_model(train_data):
    log_message("="*60)
    log_message("НАЧАЛО ДООБУЧЕНИЯ МОДЕЛИ (Бинарная классификация)")
    log_message("="*60)
    log_message(f"Модель: {MODEL_NAME}")
    log_message(f"Устройство: {device}")
    log_message(f"Batch size: {BATCH_SIZE}")
    log_message(f"Эпохи: {NUM_EPOCHS}")
    log_message(f"Learning rate: {LEARNING_RATE}")
    log_message(f"LoRA rank: {LORA_R}")
    log_message(f"FP16: {USE_FP16}")
    log_message("="*60)

    log_message("Шаг 1/6: Подготовка данных...")
    train_dataset, val_dataset = prepare_data(train_data)

    log_message("Шаг 2/6: Загрузка токенизатора...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    log_message("Шаг 3/6: Токенизация данных...")
    tokenized_train = train_dataset.map(
        lambda x: tokenize_function(x, tokenizer),
        batched=True,
        remove_columns=['text']
    )
    tokenized_val = val_dataset.map(
        lambda x: tokenize_function(x, tokenizer),
        batched=True,
        remove_columns=['text']
    )

    log_message("Шаг 4/6: Загрузка модели на GPU...")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
    )
    model = model.to(device)

    log_message("Шаг 5/6: Настройка LoRA адаптера...")
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["query", "key", "value"],
        bias="none"
    )
    model = get_peft_model(model, lora_config)

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    log_message(f"Обучаемые параметры: {trainable_params:,} / {total_params:,} ({trainable_params/total_params*100:.2f}%)")

    log_message("Шаг 6/6: Настройка обучения...")
    training_args = TrainingArguments(
        output_dir="./tmp",
        evaluation_strategy="epoch",
        logging_strategy="steps",
        logging_steps=10,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=0.01,
        save_strategy="no",
        fp16=USE_FP16,
        remove_unused_columns=False,
        report_to="none",
        gradient_checkpointing=False,
        optim="adamw_torch",
        load_best_model_at_end=False,
        metric_for_best_model="f1",
        greater_is_better=True,
        dataloader_num_workers=0,
        max_grad_norm=1.0,
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    log_message("\n" + "="*60)
    log_message("НАЧАЛО ОБУЧЕНИЯ")
    log_message("="*60)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        log_message("Память GPU очищена перед обучением")

    try:
        total_steps = max(1, len(tokenized_train) // (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS))
        log_message(f"Начинаю обучение на {NUM_EPOCHS} эпох...")
        log_message(f"Шагов на эпоху: ~{total_steps}")

        train_result = trainer.train()

        log_message("\n" + "="*60)
        log_message("ОБУЧЕНИЕ ЗАВЕРШЕНО")
        log_message("="*60)
        log_message(f"Финальная потеря: {train_result.training_loss:.4f}")

        log_message("\nОценка на валидационном наборе...")
        eval_results = trainer.evaluate()

        log_message("\n" + "="*60)
        log_message("РЕЗУЛЬТАТЫ НА ВАЛИДАЦИИ")
        log_message("="*60)
        for key, value in eval_results.items():
            if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
                if isinstance(value, float):
                    log_message(f"{key}: {value:.4f}")
                else:
                    log_message(f"{key}: {value}")

        if torch.cuda.is_available():
            memory_allocated = torch.cuda.memory_allocated() / 1e9
            memory_reserved = torch.cuda.memory_reserved() / 1e9
            log_message(f"\nИспользование памяти GPU:")
            log_message(f"  Выделено: {memory_allocated:.2f} GB")
            log_message(f"  Зарезервировано: {memory_reserved:.2f} GB")

        log_message("\n" + "="*60)
        log_message("МОДЕЛЬ ГОТОВА К ИСПОЛЬЗОВАНИЮ")
        log_message("="*60)

        return model, tokenizer, eval_results

    except Exception as e:
        log_message(f"\nОШИБКА ПРИ ОБУЧЕНИИ: {e}")
        import traceback
        traceback.print_exc()
        raise

In [ ]:
def predict_batch(model, tokenizer, texts, batch_size=32):
    log_message(f"Начинаю предсказание для {len(texts)} текстов...")

    model.eval()
    model.to(device)

    all_predictions = []
    all_probabilities = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i:i+batch_size]

            inputs = tokenizer(
                batch_texts,
                truncation=True,
                padding=True,
                max_length=MAX_LENGTH,
                return_tensors="pt"
            ).to(device)

            outputs = model(**inputs)
            logits = outputs.logits

            probs = torch.softmax(logits, dim=-1)
            predictions = torch.argmax(probs, dim=-1)

            all_predictions.extend(predictions.cpu().numpy().tolist())
            all_probabilities.extend(probs.cpu().numpy().tolist())

            if len(texts) > batch_size * 10 and (i // batch_size) % 10 == 0:
                log_message(f"  Обработано {min(i+batch_size, len(texts))}/{len(texts)} текстов")

    log_message(f"Предсказание завершено")
    return all_predictions, all_probabilities

def predict_single(model, tokenizer, text):
    model.eval()
    model.to(device)

    with torch.no_grad():
        inputs = tokenizer(
            text,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        ).to(device)

        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        prediction = torch.argmax(probs, dim=-1).item()

        toxicity_prob = probs[0, 1].item()

        return {
            'prediction': prediction,
            'probability': toxicity_prob,
            'is_toxic': prediction == 1,
            'probabilities': probs[0].cpu().numpy().tolist()
        }

In [35]:
try:
    model, tokenizer, metrics = train_lora_model(train_data)

    log_message("\n" + "="*60)
    log_message("ТЕСТИРОВАНИЕ МОДЕЛИ")
    log_message("="*60)

    test_texts = [
        "Супер, мне очень понравилось!",
        "Ты дурак и ничего не понимаешь!",
        "Спасибо за быстрый ответ",
        "Закрой уже свой рот, тупой",
        "Неплохо, но можно лучше",
        "Все вы тут конченые идиоты"
    ]

    predictions, probabilities = predict_batch(model, tokenizer, test_texts, batch_size=8)

    for i, (text, pred, probs) in enumerate(zip(test_texts, predictions, probabilities)):
        log_message(f"\nТекст {i+1}: {text[:60]}...")
        log_message(f"  Предсказание: {'ТОКСИЧНЫЙ' if pred == 1 else 'НЕ ТОКСИЧНЫЙ'}")
        log_message(f"  Вероятность класса 0 (нетоксичный): {probs[0]:.3f}")
        log_message(f"  Вероятность класса 1 (токсичный): {probs[1]:.3f}")

except Exception as e:
    log_message(f"\nОшибка: {e}")
    import traceback
    traceback.print_exc()

[10:55:32] ============================================================
[10:55:32] НАЧАЛО ДООБУЧЕНИЯ МОДЕЛИ (Бинарная классификация)
[10:55:32] ============================================================
[10:55:32] Модель: SkolkovoInstitute/russian_toxicity_classifier
[10:55:32] Устройство: cuda
[10:55:32] Batch size: 16
[10:55:32] Эпохи: 2
[10:55:32] Learning rate: 0.0002
[10:55:32] LoRA rank: 8
[10:55:32] FP16: False
[10:55:32] ============================================================
[10:55:32] Шаг 1/6: Подготовка данных...
[10:55:32] Начинаю подготовку данных...
[10:55:32] Уникальные метки в данных: [np.int64(0), np.int64(1)]
[10:55:33] Размер тренировочного набора: 139042
[10:55:33] Размер валидационного набора: 34761
[10:55:34] Распределение меток в train: 0=114230, 1=24812
[10:55:34] Распределение меток в val: 0=28349, 1=6412
[10:55:34] Шаг 2/6: Загрузка токенизатора...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[10:55:34] Шаг 3/6: Токенизация данных...


Map:   0%|          | 0/139042 [00:00<?, ? examples/s]

Map:   0%|          | 0/34761 [00:00<?, ? examples/s]

[10:56:06] Шаг 4/6: Загрузка модели на GPU...
[10:56:07] Шаг 5/6: Настройка LoRA адаптера...
[10:56:07] Обучаемые параметры: 443,906 / 178,298,884 (0.25%)
[10:56:07] Шаг 6/6: Настройка обучения...
[10:56:07] 
[10:56:07] НАЧАЛО ОБУЧЕНИЯ
[10:56:07] ============================================================
[10:56:07] Память GPU очищена перед обучением
[10:56:07] Начинаю обучение на 2 эпох...
[10:56:07] Шагов на эпоху: ~4345


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
0,0.037900,0.035504,0.987313,0.965420,0.970825,0.960075
1,0.029700,0.035626,0.987112,0.965169,0.962326,0.968029


[11:51:08] 
[11:51:08] ОБУЧЕНИЕ ЗАВЕРШЕНО
[11:51:08] ============================================================
[11:51:08] Финальная потеря: 0.0354
[11:51:08] 
Оценка на валидационном наборе...


[11:53:39] 
[11:53:39] РЕЗУЛЬТАТЫ НА ВАЛИДАЦИИ
[11:53:39] ============================================================
[11:53:39] eval_loss: 0.0356
[11:53:39] eval_accuracy: 0.9871
[11:53:39] eval_f1: 0.9652
[11:53:39] eval_precision: 0.9623
[11:53:39] eval_recall: 0.9680
[11:53:39] 
Использование памяти GPU:
[11:53:39]   Выделено: 1.47 GB
[11:53:39]   Зарезервировано: 3.43 GB
[11:53:39] 
[11:53:39] МОДЕЛЬ ГОТОВА К ИСПОЛЬЗОВАНИЮ
[11:53:39] ============================================================
[11:53:39] 
[11:53:39] ТЕСТИРОВАНИЕ МОДЕЛИ
[11:53:39] ============================================================
[11:53:39] Начинаю предсказание для 6 текстов...
[11:53:39] Предсказание завершено
[11:53:39] 
Текст 1: Супер, мне очень понравилось!...
[11:53:39]   Предсказание: НЕ ТОКСИЧНЫЙ
[11:53:39]   Вероятность класса 0 (нетоксичный): 1.000
[11:53:39]   Вероятность класса 1 (токсичный): 0.000
[11:53:39] 
Текст 2: Ты дурак и ничего не понимаешь!...
[11:53:39]   Предсказание: ТОКСИЧНЫЙ
[1

По итогам дообучения можно сделать вывод, что дальнейшее увеличение эпох точно не имеет смысла так как лос на валидаци стал немного расходиться а на трейне продолжает падать, значет мы рискуем сильно переобучиться под наш датасет. Проведем на данном этапе предикт на тестовом наборе и замери итоговый скор на кагле.

In [36]:
test_text = list(test_data['text'])

In [37]:
predictions, probabilities = predict_batch(model, tokenizer, test_text, batch_size=64)

[11:53:40] Начинаю предсказание для 74487 текстов...
[11:53:40]   Обработано 64/74487 текстов
[11:53:41]   Обработано 704/74487 текстов
[11:53:43]   Обработано 1344/74487 текстов
[11:53:44]   Обработано 1984/74487 текстов
[11:53:47]   Обработано 2624/74487 текстов
[11:53:48]   Обработано 3264/74487 текстов
[11:53:50]   Обработано 3904/74487 текстов
[11:53:52]   Обработано 4544/74487 текстов
[11:53:54]   Обработано 5184/74487 текстов
[11:53:55]   Обработано 5824/74487 текстов
[11:53:57]   Обработано 6464/74487 текстов
[11:53:59]   Обработано 7104/74487 текстов
[11:54:00]   Обработано 7744/74487 текстов
[11:54:02]   Обработано 8384/74487 текстов
[11:54:03]   Обработано 9024/74487 текстов
[11:54:05]   Обработано 9664/74487 текстов
[11:54:06]   Обработано 10304/74487 текстов
[11:54:08]   Обработано 10944/74487 текстов
[11:54:10]   Обработано 11584/74487 текстов
[11:54:11]   Обработано 12224/74487 текстов
[11:54:13]   Обработано 12864/74487 текстов
[11:54:14]   Обработано 13504/74487 тексто

In [38]:
test_id = list(test_data['id'])

In [39]:
predictions_final = pd.DataFrame({
    'id': test_id,
    'score': predictions
})

In [40]:
predictions_final_prob = pd.DataFrame({
    'id': test_id,
    'score': probabilities
})

In [41]:
predictions_final_prob

,id,score
0,152591,"[0.9999678134918213, 3.21923362207599e-05]"
1,108771,"[0.0011030265595763922, 0.9988969564437866]"
2,198853,"[0.9999518394470215, 4.813724444829859e-05]"
3,194736,"[0.9999508857727051, 4.9086618673754856e-05]"
4,88110,"[0.998389720916748, 0.0016103574307635427]"
...,...,...
74482,142159,"[0.9999291896820068, 7.086326513672248e-05]"
74483,113769,"[0.9999483823776245, 5.164354297448881e-05]"
74484,60453,"[0.9999626874923706, 3.734356869244948e-05]"
74485,166036,"[0.7626009583473206, 0.23739908635616302]"


In [ ]:
import ast
predictions_final_prob['score'] = predictions_final_prob['score'].apply(lambda x: ast.literal_eval(x)[1])

In [42]:
predictions_final.to_csv('predictions_final.csv', index=False)
predictions_final_prob.to_csv('predictions_final_prob.csv', index=False)

Итоговый roc-auc на пбуличной оценке составил 0.9981, что является на мой взгляд очень дастойным результатом, предлагаю на этом и остановить эксперименты